# Búsqueda semántica con las notas del tutorial

Este notebook descarga el código y las doce notas sintéticas del checkpoint `eb62f5d69b42a069bb4fde4c32fe918ac37a28ca`. Ejecuta su índice y sus cuatro consultas de evaluación con la revisión del modelo fijada en ese código.

El experimento corre en CPU dentro de un entorno del proyecto. La primera ejecución descarga Python, dependencias y pesos del modelo. El lockfile de Linux incluye bibliotecas de CUDA aunque la inferencia use CPU, por lo que la descarga puede ser grande.

En Colab, inicia un runtime nuevo y ejecuta las celdas en orden. Trabajaremos con el corpus público del tutorial. Colab procesa estos archivos en su infraestructura.

[Código del checkpoint](https://github.com/ricardogr07/theguildit-repos/tree/eb62f5d69b42a069bb4fde4c32fe918ac37a28ca/projects/mcp-vault-search)


In [ ]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path

CHECKPOINT = "eb62f5d69b42a069bb4fde4c32fe918ac37a28ca"
MODEL_REVISION = "e8f8c211226b894fcb81acc59f3b34ba3efd5f42"
UV_VERSION = "0.10.4"
REPOSITORY_URL = "https://github.com/ricardogr07/theguildit-repos.git"
workspace = Path(tempfile.mkdtemp(prefix="guildit-colab-"))
repository = workspace / "theguildit-repos"
project = repository / "projects" / "mcp-vault-search"
print("Carpeta temporal del experimento", workspace)


## 1. Descargar el checkpoint público

El checkout apunta al hash completo. Cada ejecución de la celda inicial crea una carpeta temporal nueva. La descarga usa Git sin un asistente de credenciales.


In [ ]:
subprocess.run(["git", "init", "--quiet", str(repository)], check=True)
subprocess.run(
    ["git", "-C", str(repository), "-c", "credential.helper=", "fetch",
     "--quiet", "--depth", "1", REPOSITORY_URL, CHECKPOINT],
    check=True,
)
subprocess.run(
    ["git", "-C", str(repository), "-c", "core.autocrlf=false",
     "checkout", "--quiet", "--detach", "FETCH_HEAD"],
    check=True,
)
actual_commit = subprocess.check_output(
    ["git", "-C", str(repository), "rev-parse", "HEAD"], text=True
).strip()
assert actual_commit == CHECKPOINT
notes = sorted((project / "examples" / "vault").glob("*.md"))
assert len(notes) == 12
print("Commit descargado", actual_commit)
print("Notas", [note.name for note in notes])
print()
print((project / "examples" / "vault" / "mcp.md").read_text(encoding="utf-8"))


## 2. Preparar las dependencias fijadas

`uv sync --locked` utiliza el lockfile del checkpoint y prepara Python 3.11 en un entorno propio del proyecto. Las llamadas siguientes ejecutan ese entorno. El kernel conserva su entorno de Colab.

Si falla una descarga, revisa el error antes de continuar. Al volver a ejecutar esta celda, uv reutiliza las descargas disponibles.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", f"uv=={UV_VERSION}"],
    check=True,
)
uv = [sys.executable, "-m", "uv"]
subprocess.run(
    [*uv, "sync", "--locked", "--extra", "semantic", "--python", "3.11"],
    cwd=project,
    check=True,
)


## 3. Crear el índice y ejecutar las cuatro consultas

El evaluador del checkpoint llama a `build_index` y `semantic_search`. El modelo corre en CPU y construye embeddings normalizados. Se comprueba que los fragmentos respeten su límite de 128 tokens, incluidos los tokens especiales.

Cada consulta pasa si la nota objetivo aparece entre las tres primeras rutas distintas y por encima del distractor designado. El reporte conserva también la búsqueda literal de la frase completa.


In [ ]:
report_path = project / "evaluate_semantic_results_colab.json"
subprocess.run(
    [*uv, "run", "--locked", "--extra", "semantic", "python",
     "evaluate_semantic.py", "--output", str(report_path)],
    cwd=project,
    check=True,
)
results = json.loads(report_path.read_text(encoding="utf-8"))
assert len(results) == 4
metadata = results[0]["evaluation_metadata"]
assert metadata["model_revision"] == MODEL_REVISION
assert len(metadata["corpus_sha256"]) == 12
assert metadata["max_chunk_tokens"] <= metadata["model_token_limit"] == 128


## 4. Revisar los resultados observados

La corrida de referencia incluida en el checkpoint logró 2/4. Esta tabla muestra lo que produjo el runtime actual y lo compara con las posiciones guardadas. Conserva los fallos al compartir el reporte.

El puntaje de similitud sirve para ordenar candidatos. La evaluación utiliza posiciones de notas distintas y el distractor de cada consulta.


In [ ]:
from IPython.display import Markdown, display

baseline = json.loads(
    (project / "evaluate_semantic_results_checkpoint_fixes.json").read_text(encoding="utf-8")
)
table = [
    "| Consulta | Objetivo | Posición actual | Posición de referencia | Distractor | Posición distractor | Resultado |",
    "| --- | --- | --- | --- | --- | --- | --- |",
]
for number, (result, reference) in enumerate(zip(results, baseline, strict=True), 1):
    assert result["query"] == reference["query"]
    table.append(
        f'| {number} | {result["expected_target"]} | {result["target_position"]} '
        f'| {reference["target_position"]} | {result["designated_distractor"]} '
        f'| {result["distractor_position"]} | {"Pasa" if result["pass"] else "Falla"} |'
    )
display(Markdown("\n".join(table)))
print("Consultas aprobadas", sum(result["pass"] for result in results), "/", len(results))
for number, result in enumerate(results, 1):
    print(number, result["query"])
    print("Coincidencias literales", result["literal_matches"])


## 5. Ubicar un fragmento en la nota original

La segunda consulta pregunta cómo encontrar algo escrito con otras palabras. Inspeccionamos su primer resultado y comprobamos que el texto devuelto esté dentro del rango de líneas de la nota descargada.


In [ ]:
match = results[1]["full_results"][0]
note_path = project / "examples" / "vault" / match["path"]
source_lines = note_path.read_text(encoding="utf-8").splitlines(keepends=True)
source_range = "".join(source_lines[match["line_start"] - 1:match["line_end"]])
assert match["text"] in source_range
print("Nota", match["path"])
print("Líneas", match["line_start"], "a", match["line_end"])
print("Similitud", match["score"])
print(source_range)


## 6. Guardar la evidencia

El reporte de esta ejecución queda en la carpeta del proyecto con el nombre `evaluate_semantic_results_colab.json`. Puedes descargarlo desde el panel de archivos de Colab, siguiendo la carpeta temporal impresa al principio.

Las huellas registran los bytes de los archivos usados en esta ejecución. El sistema operativo y los finales de línea pueden producir huellas distintas de las de la corrida de referencia en Windows.


In [ ]:
import hashlib

for filename, expected in metadata["corpus_sha256"].items():
    current = (project / "examples" / "vault" / filename).read_bytes()
    assert hashlib.sha256(current).hexdigest() == expected
for filename, expected in metadata["code_sha256"].items():
    assert hashlib.sha256((project / filename).read_bytes()).hexdigest() == expected
print("Checkpoint", CHECKPOINT)
print("Revisión del modelo", metadata["model_revision"])
print("Python del experimento", metadata["python_version"])
print("Paquetes", metadata["package_versions"])
print("Máximo de tokens por fragmento", metadata["max_chunk_tokens"])
print("Reporte", report_path)


El experimento muestra recuperación sobre un corpus pequeño. La revisión humana de relevancia sigue siendo parte de la evaluación, especialmente en las consultas 1 y 4.

El servidor MCP utiliza este mismo código desde un cliente local. El tutorial retoma después la conexión del cliente, la lectura de las notas y la reconstrucción del índice tras editar los archivos.
